In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold
import os
import random

from transformers import (
    AutoTokenizer
)

from data.datasets.Smiles_Dataset import SMILESDataset
from data.preprocessing.pipeline import Pipeline
from models.BERT import ChemBERTaClassifier
from utils.paths import WEIGHTS_DIR

/home/smirnov@dohod.local/Desktop/thortex-hackaton/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [3]:
# Гиперпараметры
k_folds = 5
epochs = 50
batch_size = 32
lr = 2e-5
dropout = 0.3
patience = 10
model_name = "DeepChem/ChemBERTa-77M-MTR"
max_length = 128
save_dir_base = os.path.join(WEIGHTS_DIR, 'chemberta_cv')
checkpoint_prefix = "chemberta_fold"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed()

In [4]:
pipeline = Pipeline(feature_type="morgan", download=False)

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s]

Extracting raw features: 100%|██████████| 888/888 [00:02<00:00, 399.26it/s]


In [5]:
from rdkit import Chem

def augment_rare_classes(df, rare_classes, augment_factor=5):
    augmented = []
    for _, row in df.iterrows():
        if row["taste_cluster"] in rare_classes:
            for _ in range(augment_factor):
                mol = Chem.MolFromSmiles(row["SMILES"])
                if mol is not None:
                    new_smiles = Chem.MolToSmiles(mol, doRandom=True)
                    augmented.append({
                        "SMILES": new_smiles,
                        "taste_cluster": row["taste_cluster"],
                    })
    return pd.DataFrame(augmented)

In [6]:
def train_model(
    train_df,
    val_df,
    epochs=20,
    batch_size=64,
    lr=2e-5,
    dropout=0.3,
    patience=5,
    model_name="DeepChem/ChemBERTa-77M-MTR",
    max_length=128,
    checkpoint_path="best_chemberta.pth",
    save_dir_path=".",
    label_names=None
):
    set_seed()
    os.makedirs(save_dir_path, exist_ok=True)

    num_labels = train_df["taste_cluster"].nunique()

    if label_names is None:
        label_names = [str(i) for i in range(num_labels)]
    else:
        assert len(label_names) == num_labels, "label_names length must match number of classes"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = SMILESDataset(train_df, tokenizer, max_length=max_length, mode="train")
    val_dataset = SMILESDataset(val_df, tokenizer, max_length=max_length, mode="train")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = ChemBERTaClassifier(num_labels=num_labels, model_name=model_name, dropout=dropout).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(logits, labels)
                val_losses.append(loss.item())

                preds = torch.argmax(logits, dim=1)
                all_preds.append(preds.cpu().numpy())
                all_targets.append(labels.cpu().numpy())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

        val_preds = np.concatenate(all_preds)
        val_targets = np.concatenate(all_targets)
        macro_f1 = f1_score(val_targets, val_preds, average='macro')

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save(model.state_dict(), os.path.join(save_dir_path, checkpoint_path))
            patience_counter = 0
            print(f"  → New best model saved! f1={best_f1:.5f}")
        else:
            patience_counter += 1
            print(f"  (patience {patience_counter}/{patience})")
            if patience_counter >= patience:
                print("\nEarly stopping triggered!")
                break

    # -------------------------
    # Final evaluation
    # -------------------------
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    print("\nFinal macro F1 on validation:", macro_f1)

    # -------------------------
    # Classification report
    # -------------------------
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT (Validation Set)")
    print("=" * 60)
    report = classification_report(
        all_targets,
        all_preds,
        target_names=label_names,
        zero_division=0
    )
    print(report)

    return model, macro_f1, best_f1

In [7]:
def predict(model, test_df, batch_size=64, model_name="DeepChem/ChemBERTa-77M-MTR", max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    test_dataset = SMILESDataset(test_df, tokenizer, max_length=max_length, mode="test")
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    all_ids = []
    all_preds = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            ids = batch["id"]

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_ids.extend(ids)
            all_preds.extend(preds.cpu().numpy())

    pred_df = pd.DataFrame({"Pubchem_ID": all_ids, "taste_cluster": all_preds})
    return pred_df

In [8]:
# Данные
train_df, test_df = pipeline.get_raw_data()
train_df = train_df.copy()
train_df["taste_cluster"] = train_df["taste_cluster"].astype(int)

# Стратифицированный KFold
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["taste_cluster"])):
    print(f"\n{'='*50}")
    print(f" FOLD {fold + 1}/{k_folds}")
    print(f"{'='*50}")

    # Разделение
    train_fold_orig = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    # Аугментация ТОЛЬКО трейна
    aug_df = augment_rare_classes(train_fold_orig, rare_classes=[5, 6], augment_factor=8)
    aug_df2 = augment_rare_classes(train_fold_orig, rare_classes=[7, 2], augment_factor=3)
    aug_df3 = augment_rare_classes(train_fold_orig, rare_classes=[1, 3, 4], augment_factor=2)

    train_fold_balanced = pd.concat([train_fold_orig, aug_df, aug_df2, aug_df3], ignore_index=True)
    train_fold_balanced["taste_cluster"] = train_fold_balanced["taste_cluster"].astype(int)

    # Директория для фолда
    save_dir_fold = os.path.join(save_dir_base, f"fold_{fold + 1}")
    checkpoint_path = f"{checkpoint_prefix}_fold_{fold + 1}.pth"

    # Обучение модели на фолде
    model, val_f1, _ = train_model(
        train_df=train_fold_balanced,
        val_df=val_fold,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        dropout=dropout,
        patience=patience,
        model_name=model_name,
        max_length=max_length,
        checkpoint_path=checkpoint_path,
        save_dir_path=save_dir_fold,
        label_names=None
    )

    fold_results.append({
        "fold": fold + 1,
        "val_f1": val_f1,
        "model_path": os.path.join(save_dir_fold, checkpoint_path),
        "model": model  # опционально, если память позволяет
    })

    print(f"Fold {fold + 1} finished. Val F1: {val_f1:.5f}\n")

# Средний результат
mean_f1 = np.mean([r["val_f1"] for r in fold_results])
std_f1 = np.std([r["val_f1"] for r in fold_results])
print(f"\n{'='*60}")
print(f"K-FOLD CROSS-VALIDATION RESULTS ({k_folds} folds)")
print(f"Mean Macro F1: {mean_f1:.5f} ± {std_f1:.5f}")
print(f"{'='*60}")


 FOLD 1/5


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 | train_loss=2.03138 | val_loss=2.00195
  → New best model saved! f1=0.13285
Epoch 2/50 | train_loss=1.83909 | val_loss=1.78155
  → New best model saved! f1=0.23215
Epoch 3/50 | train_loss=1.59869 | val_loss=1.60430
  → New best model saved! f1=0.36360
Epoch 4/50 | train_loss=1.47165 | val_loss=1.53669
  (patience 1/10)
Epoch 5/50 | train_loss=1.42019 | val_loss=1.52501
  → New best model saved! f1=0.40928
Epoch 6/50 | train_loss=1.37634 | val_loss=1.48886
  (patience 1/10)
Epoch 7/50 | train_loss=1.33105 | val_loss=1.48407
  (patience 2/10)
Epoch 8/50 | train_loss=1.31898 | val_loss=1.46691
  → New best model saved! f1=0.42703
Epoch 9/50 | train_loss=1.29772 | val_loss=1.45962
  → New best model saved! f1=0.44396
Epoch 10/50 | train_loss=1.27390 | val_loss=1.45212
  → New best model saved! f1=0.45048
Epoch 11/50 | train_loss=1.25504 | val_loss=1.44978
  (patience 1/10)
Epoch 12/50 | train_loss=1.23541 | val_loss=1.44598
  (patience 2/10)
Epoch 13/50 | train_loss=1.21910 | v

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 | train_loss=2.02902 | val_loss=2.00909
  → New best model saved! f1=0.13566
Epoch 2/50 | train_loss=1.82426 | val_loss=1.80740
  → New best model saved! f1=0.20212
Epoch 3/50 | train_loss=1.58366 | val_loss=1.63874
  → New best model saved! f1=0.29837
Epoch 4/50 | train_loss=1.46143 | val_loss=1.58832
  → New best model saved! f1=0.31847
Epoch 5/50 | train_loss=1.39838 | val_loss=1.54719
  → New best model saved! f1=0.37722
Epoch 6/50 | train_loss=1.35410 | val_loss=1.51115
  → New best model saved! f1=0.41279
Epoch 7/50 | train_loss=1.32807 | val_loss=1.52156
  (patience 1/10)
Epoch 8/50 | train_loss=1.30515 | val_loss=1.49280
  (patience 2/10)
Epoch 9/50 | train_loss=1.28205 | val_loss=1.48103
  (patience 3/10)
Epoch 10/50 | train_loss=1.26094 | val_loss=1.47901
  (patience 4/10)
Epoch 11/50 | train_loss=1.24897 | val_loss=1.48826
  (patience 5/10)
Epoch 12/50 | train_loss=1.23473 | val_loss=1.46562
  (patience 6/10)
Epoch 13/50 | train_loss=1.21197 | val_loss=1.46696
  (

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 | train_loss=2.02683 | val_loss=2.01210
  → New best model saved! f1=0.13137
Epoch 2/50 | train_loss=1.80156 | val_loss=1.84192
  → New best model saved! f1=0.21176
Epoch 3/50 | train_loss=1.54567 | val_loss=1.72530
  → New best model saved! f1=0.30000
Epoch 4/50 | train_loss=1.43068 | val_loss=1.68080
  → New best model saved! f1=0.31925
Epoch 5/50 | train_loss=1.37436 | val_loss=1.65870
  → New best model saved! f1=0.36184
Epoch 6/50 | train_loss=1.34862 | val_loss=1.62897
  (patience 1/10)
Epoch 7/50 | train_loss=1.30872 | val_loss=1.60104
  → New best model saved! f1=0.40186
Epoch 8/50 | train_loss=1.28658 | val_loss=1.60161
  (patience 1/10)
Epoch 9/50 | train_loss=1.25993 | val_loss=1.58490
  → New best model saved! f1=0.40274
Epoch 10/50 | train_loss=1.25112 | val_loss=1.56000
  (patience 1/10)
Epoch 11/50 | train_loss=1.23186 | val_loss=1.56071
  (patience 2/10)
Epoch 12/50 | train_loss=1.21914 | val_loss=1.55860
  → New best model saved! f1=0.41911
Epoch 13/50 | tra

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 | train_loss=2.02762 | val_loss=2.00265
  → New best model saved! f1=0.14254
Epoch 2/50 | train_loss=1.82034 | val_loss=1.78391
  → New best model saved! f1=0.20110
Epoch 3/50 | train_loss=1.57713 | val_loss=1.61841
  → New best model saved! f1=0.28876
Epoch 4/50 | train_loss=1.46967 | val_loss=1.56555
  → New best model saved! f1=0.35524
Epoch 5/50 | train_loss=1.40457 | val_loss=1.54094
  → New best model saved! f1=0.35777
Epoch 6/50 | train_loss=1.37213 | val_loss=1.52206
  → New best model saved! f1=0.38796
Epoch 7/50 | train_loss=1.33804 | val_loss=1.50374
  (patience 1/10)
Epoch 8/50 | train_loss=1.32009 | val_loss=1.49401
  → New best model saved! f1=0.40433
Epoch 9/50 | train_loss=1.29412 | val_loss=1.49873
  (patience 1/10)
Epoch 10/50 | train_loss=1.26977 | val_loss=1.49603
  (patience 2/10)
Epoch 11/50 | train_loss=1.25676 | val_loss=1.48974
  → New best model saved! f1=0.40634
Epoch 12/50 | train_loss=1.22914 | val_loss=1.49491
  (patience 1/10)
Epoch 13/50 | tra

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 | train_loss=2.03071 | val_loss=2.00648
  → New best model saved! f1=0.13426
Epoch 2/50 | train_loss=1.82745 | val_loss=1.78941
  → New best model saved! f1=0.22393
Epoch 3/50 | train_loss=1.60770 | val_loss=1.63044
  → New best model saved! f1=0.33875
Epoch 4/50 | train_loss=1.48175 | val_loss=1.58284
  → New best model saved! f1=0.36881
Epoch 5/50 | train_loss=1.43404 | val_loss=1.52518
  (patience 1/10)
Epoch 6/50 | train_loss=1.38135 | val_loss=1.50826
  (patience 2/10)
Epoch 7/50 | train_loss=1.34006 | val_loss=1.49124
  → New best model saved! f1=0.38633
Epoch 8/50 | train_loss=1.32522 | val_loss=1.47400
  (patience 1/10)
Epoch 9/50 | train_loss=1.29121 | val_loss=1.46214
  (patience 2/10)
Epoch 10/50 | train_loss=1.27316 | val_loss=1.46309
  (patience 3/10)
Epoch 11/50 | train_loss=1.25837 | val_loss=1.45170
  (patience 4/10)
Epoch 12/50 | train_loss=1.23975 | val_loss=1.44475
  (patience 5/10)
Epoch 13/50 | train_loss=1.21205 | val_loss=1.43789
  → New best model sav

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

# --- Восстановление моделей из чекпоинтов ---
models = []
k_folds = 5

for fold in range(1, k_folds + 1):
    model = ChemBERTaClassifier(num_labels=8, model_name=model_name, dropout=0.3).to(device)
    checkpoint_path = os.path.join(save_dir_base, f"fold_{fold}", f"chemberta_fold_fold_{fold}.pth")
    
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    models.append(model)
    print(f"Loaded model from fold {fold}")

# --- Функция ансамблевого предсказания ---
def ensemble_predict(models, test_df, tokenizer, max_length=128, device=device):
    all_logits = []
    
    for model in models:
        model.eval()
        logits_list = []
        for _, row in test_df.iterrows():
            inputs = tokenizer(
                row["SMILES"],
                return_tensors="pt",
                padding="max_length",
                max_length=max_length,
                truncation=True
            ).to(device)
            
            with torch.no_grad():
                outputs = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"]
                )
                # outputs — это logits: shape [1, num_labels]
                logits_list.append(outputs.cpu().numpy())
        
        # Собираем логиты для всего теста от одной модели
        all_logits.append(np.vstack(logits_list))  # shape: (N_test, num_labels)
    
    # Усреднение по моделям
    avg_logits = np.mean(all_logits, axis=0)  # shape: (N_test, num_labels)
    final_preds = np.argmax(avg_logits, axis=1)
    return final_preds, avg_logits

# --- Запуск инференса ---
print("Running ensemble prediction on test set...")
test_preds, _ = ensemble_predict(models, test_df, tokenizer, max_length=max_length, device=device)

# --- Сохранение предсказаний (опционально) ---
output_path = os.path.join(WEIGHTS_DIR, "test_predictions_ensemble.csv")
if "Pubchem_ID" in test_df.columns:
    submission_df = pd.DataFrame({
        "taste_cluster": test_preds
    })
else:
    submission_df = pd.DataFrame({
        "taste_cluster": test_preds
    })
submission_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

# --- Оценка на тесте (если есть истинные метки) ---
if "taste_cluster" in test_df.columns:
    true_labels = test_df["taste_cluster"].values
    test_f1 = f1_score(true_labels, test_preds, average="macro")
    print(f"\n✅ Test Macro F1 (Ensemble of {k_folds} models): {test_f1:.5f}")
else:
    print("\n⚠️ No ground truth labels in test_df — skipping evaluation.")

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model from fold 1


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model from fold 2


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model from fold 3


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model from fold 4


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model from fold 5
Running ensemble prediction on test set...
Predictions saved to /home/smirnov@dohod.local/Desktop/thortex-hackaton/weights/test_predictions_ensemble.csv

⚠️ No ground truth labels in test_df — skipping evaluation.
